# ADHD / EEG — Part 2: richer features, patient-level models, robust CV

Main notebook is methodologically sound, so this **adds to it** rather than
changing it. Three honest levers for better and more defensible results:

1. **Richer features** — connectivity, frontal alpha asymmetry, extra band ratios,
   spectral edge/peak frequency, permutation entropy (all ADHD/EEG literature).
2. **A new patient-level model** — one row per patient (mean+std of features), which
   targets the diagnosis unit directly and tends to be more stable.
3. **More models + repeated CV** — LightGBM/CatBoost/XGBoost added to the zoo, and
   repeated patient-grouped CV so the headline number is mean ± std, not one split.

Everything keeps the same anti-leak protocol: patient-grouped splitting, scaling and
feature selection fit **inside** CV. Expected honest range on this dataset stays
~0.80–0.90 patient-level AUC; a sudden jump to ~0.95+ is a leakage red flag, not a win.

## Part 2 · Setup
Runs **alongside** my existing notebook without changing it. Same pickle files,
same honest protocol (patient-grouped splitting, scaling inside CV). Optional
gradient boosters (LightGBM / CatBoost / XGBoost) are used automatically if installed.

In [1]:
# =====================================================================
# ADHD / EEG — PART 2: richer features, patient-level models, robust CV
# =====================================================================
# This runs ALONGSIDE the existing notebook (it does not change it). It
# re-loads the same pickle files, builds a RICHER feature set, adds a whole
# new *patient-level* modeling approach, expands the model zoo, and reports
# results with REPEATED grouped cross-validation so the numbers are stable
# and defensible. The honest protocol (patient-grouped splitting, scaling
# inside CV) is preserved throughout.

import math, pickle, warnings
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.signal import welch
from scipy.stats import skew, kurtosis, entropy as spectral_entropy_fn

from sklearn.model_selection import (StratifiedGroupKFold, StratifiedKFold,
                                     RepeatedStratifiedKFold, train_test_split as _tts)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier, GradientBoostingClassifier,
                              StackingClassifier)
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATHS = {"full": "eeg_dataset_full_zscore.pkl",
              "simple": "eeg_dataset_simple_zscore.pkl"}
PRIMARY_PIPELINE = "full"

# optional gradient-boosting libraries (used if installed)
OPT = {}
try:
    from lightgbm import LGBMClassifier; OPT["LightGBM"] = LGBMClassifier
except Exception: pass
try:
    from catboost import CatBoostClassifier; OPT["CatBoost"] = CatBoostClassifier
except Exception: pass
try:
    from xgboost import XGBClassifier; OPT["XGBoost"] = XGBClassifier
except Exception: pass
print("optional boosters available:", list(OPT.keys()) or "none")


optional boosters available: ['LightGBM', 'CatBoost', 'XGBoost']


## Load the same preprocessed epochs

In [2]:
def load_epoch_dataset(path):
    with open(path, "rb") as f:
        d = pickle.load(f)
    return d

datasets = {}
for name, path in DATA_PATHS.items():
    try:
        datasets[name] = load_epoch_dataset(path)
        d = datasets[name]
        print(f"Loaded {name}: epochs {d['epochs'].shape}, "
              f"{len(np.unique(d['patient_ids']))} patients")
    except FileNotFoundError:
        print(f"[!] missing {path} - skipping {name}")

sfreq = datasets[PRIMARY_PIPELINE].get("sfreq", 128)
channel_names = list(datasets[PRIMARY_PIPELINE].get(
    "channel_names", [f"ch{i}" for i in range(datasets[PRIMARY_PIPELINE]["epochs"].shape[1])]))


Loaded full: epochs (16749, 15, 256), 121 patients
Loaded simple: epochs (16749, 15, 256), 121 patients


## Richer features
Keeps every original per-channel feature and **adds**, per channel: extra band ratios
(theta/alpha, alpha/beta, a delta+theta vs alpha+beta *slowing ratio*), spectral edge
& peak frequency, and **permutation entropy** (nonlinear complexity). Then per epoch:
**inter-channel connectivity** (each channel's mean |correlation| to the others, plus
global mean/std) and **frontal alpha asymmetry** for standard 10–20 pairs. All are
established ADHD/EEG markers. Roughly doubles the feature count — worthwhile because
the models are regularised and evaluated honestly.

In [3]:
# Richer feature extraction. Keeps the original per-channel spectral/Hjorth/moment
# features and ADDS: extra band ratios, spectral edge & peak frequency, permutation
# entropy (nonlinear complexity), inter-channel connectivity summaries, and frontal
# alpha asymmetry - all established in the ADHD/EEG literature.

BANDS = {"delta": (1, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 40)}
_trapz = getattr(np, "trapezoid", None) or np.trapz

def hjorth_params(x):
    dx = np.diff(x); ddx = np.diff(dx)
    var_x, var_dx, var_ddx = np.var(x), np.var(dx), np.var(ddx)
    mob = np.sqrt(var_dx / var_x) if var_x > 0 else 0.0
    comp = (np.sqrt(var_ddx / var_dx) / mob) if (var_dx > 0 and mob > 0) else 0.0
    return var_x, mob, comp

def perm_entropy(x, order=3, delay=1):
    x = np.asarray(x, float); n = len(x); m = n - (order - 1) * delay
    if m <= 1: return 0.0
    emb = np.empty((m, order))
    for dd in range(order):
        emb[:, dd] = x[dd * delay: dd * delay + m]
    perms = np.argsort(emb, axis=1)
    _, counts = np.unique(perms, axis=0, return_counts=True)
    p = counts / counts.sum()
    ent = -np.sum(p * np.log2(p))
    return ent / np.log2(math.factorial(order))

def extract_channel_features(x, sfreq):
    freqs, psd = welch(x, fs=sfreq, nperseg=min(len(x), 128))
    total = _trapz(psd, freqs) + 1e-12
    powers = {}
    for band, (lo, hi) in BANDS.items():
        mask = (freqs >= lo) & (freqs < hi)
        powers[band] = _trapz(psd[mask], freqs[mask]) if mask.any() else 0.0
    rel = {f"{b}_rel": p / total for b, p in powers.items()}
    psd_norm = psd / (psd.sum() + 1e-12)
    spec_ent = spectral_entropy_fn(psd_norm + 1e-12)
    act, mob, comp = hjorth_params(x)
    # spectral edge (95%) and peak frequency within 1-40 Hz
    band_mask = (freqs >= 1) & (freqs <= 40)
    fb, pb = freqs[band_mask], psd[band_mask]
    if len(pb) and pb.sum() > 0:
        cum = np.cumsum(pb) / pb.sum()
        sef95 = float(fb[np.searchsorted(cum, 0.95).clip(0, len(fb) - 1)])
        peakf = float(fb[np.argmax(pb)])
    else:
        sef95 = peakf = 0.0
    b = powers
    feats = {
        **{f"{k}_abs": v for k, v in b.items()}, **rel,
        "theta_beta_ratio": b["theta"] / (b["beta"] + 1e-12),
        "theta_alpha_ratio": b["theta"] / (b["alpha"] + 1e-12),
        "alpha_beta_ratio": b["alpha"] / (b["beta"] + 1e-12),
        "slowing_ratio": (b["delta"] + b["theta"]) / (b["alpha"] + b["beta"] + 1e-12),
        "total_power": total, "spectral_entropy": spec_ent,
        "spectral_edge_95": sef95, "peak_freq": peakf,
        "hjorth_activity": act, "hjorth_mobility": mob, "hjorth_complexity": comp,
        "perm_entropy": perm_entropy(x),
        "std": np.std(x), "skew": skew(x), "kurtosis": kurtosis(x),
    }
    return feats

# frontal/lateral pairs for alpha asymmetry (matched case-insensitively; 10-20 aliases)
_ASym_PAIRS = [("F3", "F4"), ("F7", "F8"), ("C3", "C4"), ("P3", "P4"),
               ("O1", "O2"), ("T3", "T4"), ("T7", "T8"), ("Fp1", "Fp2")]

def _chan_index(names):
    return {n.lower(): i for i, n in enumerate(names)}

def connectivity_and_asym(epoch, ch_names, ch_alpha_rel):
    """Epoch-level features: per-channel mean |correlation| to other channels,
    global connectivity mean/std, and frontal alpha asymmetry for standard pairs."""
    with np.errstate(all="ignore"):
        C = np.corrcoef(epoch)
    C = np.nan_to_num(C)
    absC = np.abs(C)
    np.fill_diagonal(absC, np.nan)
    per_ch = np.nanmean(absC, axis=1)
    feats = {f"{ch_names[i]}_conn": float(per_ch[i]) for i in range(len(ch_names))}
    feats["conn_global_mean"] = float(np.nanmean(absC))
    feats["conn_global_std"] = float(np.nanstd(absC))
    idx = _chan_index(ch_names)
    for a, b in _ASym_PAIRS:
        ia, ib = idx.get(a.lower()), idx.get(b.lower())
        if ia is not None and ib is not None:
            la, lb = ch_alpha_rel[ia] + 1e-12, ch_alpha_rel[ib] + 1e-12
            feats[f"alpha_asym_{a}{b}"] = float(np.log(lb) - np.log(la))
    return feats

def build_feature_matrix_rich(epochs, ch_names, sfreq, verbose_every=1000):
    rows = []
    for i in range(epochs.shape[0]):
        row, alpha_rel = {}, np.zeros(len(ch_names))
        for c, cn in enumerate(ch_names):
            cf = extract_channel_features(epochs[i, c, :], sfreq)
            alpha_rel[c] = cf["alpha_rel"]
            for k, v in cf.items():
                row[f"{cn}_{k}"] = v
        row.update(connectivity_and_asym(epochs[i], ch_names, alpha_rel))
        rows.append(row)
        if verbose_every and (i + 1) % verbose_every == 0:
            print(f"  {i + 1}/{epochs.shape[0]} epochs")
    return pd.DataFrame(rows)

feature_cache = {}
for name, d in datasets.items():
    print(f"Rich features for '{name}' ({d['epochs'].shape[0]} epochs)...")
    fdf = build_feature_matrix_rich(d["epochs"], channel_names, d.get("sfreq", sfreq))
    fdf["label"] = d["labels"]; fdf["patient_id"] = d["patient_ids"]
    feature_cache[name] = fdf
    print(f"  -> {fdf.shape}")


Rich features for 'full' (16749 epochs)...
  1000/16749 epochs
  2000/16749 epochs
  3000/16749 epochs
  4000/16749 epochs
  5000/16749 epochs
  6000/16749 epochs
  7000/16749 epochs
  8000/16749 epochs
  9000/16749 epochs
  10000/16749 epochs
  11000/16749 epochs
  12000/16749 epochs
  13000/16749 epochs
  14000/16749 epochs
  15000/16749 epochs
  16000/16749 epochs
  -> (16749, 400)
Rich features for 'simple' (16749 epochs)...
  1000/16749 epochs
  2000/16749 epochs
  3000/16749 epochs
  4000/16749 epochs
  5000/16749 epochs
  6000/16749 epochs
  7000/16749 epochs
  8000/16749 epochs
  9000/16749 epochs
  10000/16749 epochs
  11000/16749 epochs
  12000/16749 epochs
  13000/16749 epochs
  14000/16749 epochs
  15000/16749 epochs
  16000/16749 epochs
  -> (16749, 400)


## Canonical patient split (identical to the main notebook)

In [4]:
# Same canonical patient split logic as the main notebook (identical seed/method),
# so results are comparable.
primary_d = datasets[PRIMARY_PIPELINE]
pl = (pd.DataFrame({"patient_id": primary_d["patient_ids"], "label": primary_d["labels"]})
      .drop_duplicates("patient_id").reset_index(drop=True))
TRAIN_PIDS, TEST_PIDS = _tts(pl["patient_id"].values, test_size=0.2,
                             stratify=pl["label"].values, random_state=RANDOM_STATE)
TRAIN_PIDS, TEST_PIDS = set(TRAIN_PIDS), set(TEST_PIDS)

def get_X_y_groups(fdf):
    X = fdf.drop(columns=["label", "patient_id"])
    return X, fdf["label"].values, fdf["patient_id"].values

def evaluate_predictions(y, pred, proba):
    return {"accuracy": accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "roc_auc": roc_auc_score(y, proba) if len(np.unique(y)) > 1 else float("nan"),
            "mcc": matthews_corrcoef(y, pred)}


## Expanded window-level model zoo
LogReg, RandomForest, ExtraTrees, HistGB, SVM — **plus LightGBM / CatBoost / XGBoost**
when available — each scaled inside patient-grouped CV, then the best is evaluated on
the held-out patients at window and patient level.

In [5]:
# Expanded WINDOW-LEVEL model zoo, honest patient-grouped CV, evaluated on the
# held-out patients (window- and patient-level).
def make_window_models(y_train):
    pos_w = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    m = {
        "LogReg": Pipeline([("sc", StandardScaler()),
                            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                                       random_state=RANDOM_STATE))]),
        "RandomForest": Pipeline([("sc", StandardScaler()),
                            ("clf", RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                                           random_state=RANDOM_STATE, n_jobs=-1))]),
        "ExtraTrees": Pipeline([("sc", StandardScaler()),
                            ("clf", ExtraTreesClassifier(n_estimators=500, class_weight="balanced",
                                                         random_state=RANDOM_STATE, n_jobs=-1))]),
        "HistGB": Pipeline([("sc", StandardScaler()),
                            ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE))]),
        "SVM_RBF": Pipeline([("sc", StandardScaler()),
                            ("clf", SVC(kernel="rbf", probability=True, class_weight="balanced",
                                        random_state=RANDOM_STATE))]),
    }
    if "LightGBM" in OPT:
        m["LightGBM"] = Pipeline([("sc", StandardScaler()),
            ("clf", OPT["LightGBM"](n_estimators=400, learning_rate=0.03, num_leaves=31,
                                    subsample=0.8, colsample_bytree=0.8, class_weight="balanced",
                                    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1))])
    if "CatBoost" in OPT:
        m["CatBoost"] = Pipeline([("sc", StandardScaler()),
            ("clf", OPT["CatBoost"](iterations=500, depth=6, learning_rate=0.03,
                                    class_weights=[1.0, pos_w], random_seed=RANDOM_STATE, verbose=0))])
    if "XGBoost" in OPT:
        m["XGBoost"] = Pipeline([("sc", StandardScaler()),
            ("clf", OPT["XGBoost"](n_estimators=400, max_depth=4, learning_rate=0.03,
                                   subsample=0.8, scale_pos_weight=pos_w, eval_metric="logloss",
                                   random_state=RANDOM_STATE, n_jobs=-1))])
    return m

def grouped_cv_auc(pipe, X, y, groups, n_splits=5):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs = []
    for tr, va in cv.split(X, y, groups):
        pipe.fit(X[tr], y[tr])
        p = pipe.predict_proba(X[va])[:, 1]
        aucs.append(roc_auc_score(y[va], p) if len(np.unique(y[va])) > 1 else np.nan)
    return float(np.nanmean(aucs)), float(np.nanstd(aucs))

def patient_aggct(groups_te, y_te, proba):
    df = pd.DataFrame({"pid": groups_te, "y": y_te, "p": proba})
    agg = df.groupby("pid").agg(y=("y", "first"), p=("p", "mean")).reset_index()
    return evaluate_predictions(agg["y"].values, (agg["p"] >= 0.5).astype(int), agg["p"].values)

def run_window_zoo(pipeline_name):
    X, y, g = get_X_y_groups(feature_cache[pipeline_name])
    Xv = X.values
    trm, tem = np.isin(g, list(TRAIN_PIDS)), np.isin(g, list(TEST_PIDS))
    models = make_window_models(y[trm])
    ranked = []
    for nm, pipe in models.items():
        mu, sd = grouped_cv_auc(pipe, Xv[trm], y[trm], g[trm])
        ranked.append((nm, mu, sd))
    ranked.sort(key=lambda r: r[1], reverse=True)
    print(f"[{pipeline_name}] window-level grouped-CV ROC-AUC:")
    for nm, mu, sd in ranked:
        print(f"   {nm:14s} {mu:.3f} +/- {sd:.3f}")
    best = ranked[0][0]
    # held-out evaluation with the best model
    models[best].fit(Xv[trm], y[trm])
    proba = models[best].predict_proba(Xv[tem])[:, 1]
    win = evaluate_predictions(y[tem], (proba >= 0.5).astype(int), proba)
    pat = patient_aggct(g[tem], y[tem], proba)
    return best, ranked, win, pat, models[best]


## New approach — patient-level modeling
Each patient becomes **one row**: the mean and std of every window feature across their
epochs. This models the diagnosis unit directly and is usually more stable. Because
n ≈ 121, we use `SelectKBest` + strong regularisation inside CV and **repeated**
stratified CV (5×5) for a robust estimate.

In [6]:
# NEW APPROACH — patient-level modeling. Each patient becomes ONE row: the mean and
# std of every window feature across that patient's epochs. Models the diagnosis unit
# directly and is usually more stable. n is only ~121, so we use SelectKBest + strong
# regularisation inside CV, and REPEATED stratified CV for a robust estimate.
def build_patient_table(fdf):
    feats = fdf.drop(columns=["label", "patient_id"])
    g = fdf["patient_id"].values
    agg_mean = feats.groupby(g).mean()
    agg_std = feats.groupby(g).std().fillna(0.0)
    agg_mean.columns = [f"{c}_mean" for c in agg_mean.columns]
    agg_std.columns = [f"{c}_std" for c in agg_std.columns]
    Xp = pd.concat([agg_mean, agg_std], axis=1)
    lab = pd.DataFrame({"patient_id": fdf["patient_id"].values, "label": fdf["label"].values}) \
            .drop_duplicates("patient_id").set_index("patient_id")["label"]
    yp = lab.loc[Xp.index].values
    return Xp, yp, Xp.index.values

def make_patient_models(k=40):
    base = {
        "LogReg_L2": Pipeline([("sc", StandardScaler()), ("sel", SelectKBest(f_classif, k=k)),
            ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", C=0.5,
                                       random_state=RANDOM_STATE))]),
        "SVM_RBF": Pipeline([("sc", StandardScaler()), ("sel", SelectKBest(f_classif, k=k)),
            ("clf", SVC(kernel="rbf", probability=True, class_weight="balanced",
                        random_state=RANDOM_STATE))]),
        "ExtraTrees": Pipeline([("sc", StandardScaler()),
            ("clf", ExtraTreesClassifier(n_estimators=600, max_depth=None, min_samples_leaf=2,
                                         class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))]),
        "HistGB": Pipeline([("sc", StandardScaler()),
            ("clf", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                                   random_state=RANDOM_STATE))]),
    }
    if "LightGBM" in OPT:
        base["LightGBM"] = Pipeline([("sc", StandardScaler()),
            ("clf", OPT["LightGBM"](n_estimators=300, learning_rate=0.03, num_leaves=15,
                                    subsample=0.8, colsample_bytree=0.6, class_weight="balanced",
                                    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1))])
    return base

def repeated_patient_cv(pipe, Xp, yp, n_splits=5, n_repeats=5):
    Xv = Xp.values if hasattr(Xp, "values") else Xp
    rcv = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=RANDOM_STATE)
    aucs, accs = [], []
    for tr, va in rcv.split(Xv, yp):
        from sklearn.base import clone
        p = clone(pipe); p.fit(Xv[tr], yp[tr])
        pr = p.predict_proba(Xv[va])[:, 1]
        aucs.append(roc_auc_score(yp[va], pr) if len(np.unique(yp[va])) > 1 else np.nan)
        accs.append(accuracy_score(yp[va], (pr >= 0.5).astype(int)))
    return float(np.nanmean(aucs)), float(np.nanstd(aucs)), float(np.mean(accs))

def run_patient_zoo(pipeline_name):
    Xp, yp, pids = build_patient_table(feature_cache[pipeline_name])
    k = min(40, Xp.shape[1])
    models = make_patient_models(k=k)
    print(f"[{pipeline_name}] patient-level repeated CV (5x5): {Xp.shape[0]} patients, {Xp.shape[1]} feats")
    ranked = []
    for nm, pipe in models.items():
        mu, sd, acc = repeated_patient_cv(pipe, Xp, yp)
        ranked.append((nm, mu, sd, acc))
    ranked.sort(key=lambda r: r[1], reverse=True)
    for nm, mu, sd, acc in ranked:
        print(f"   {nm:12s} AUC {mu:.3f} +/- {sd:.3f} | acc {acc:.3f}")
    # held-out patients (same split) with the best patient-level model
    best = ranked[0][0]
    trm = np.isin(pids, list(TRAIN_PIDS)); tem = np.isin(pids, list(TEST_PIDS))
    from sklearn.base import clone
    p = clone(models[best]); p.fit(Xp.values[trm], yp[trm])
    pr = p.predict_proba(Xp.values[tem])[:, 1]
    held = evaluate_predictions(yp[tem], (pr >= 0.5).astype(int), pr)
    return best, ranked, held, (Xp, yp, pids)


## Robust repeated grouped CV
Your main notebook reports on a single 80/20 split (~24 test patients), which is
high-variance. Here the best window model is run through **repeated** patient-grouped
CV and the patient-level AUC/accuracy are reported as mean ± std — the number to quote
in a report, because it doesn't hinge on one lucky split.

In [7]:
# Robust repeated grouped CV for the best window-level model: patient-level AUC/acc
# aggregated within each fold, averaged over repeats -> the defensible headline number.
def repeated_grouped_patient_cv(pipe, X, y, g, n_splits=5, n_repeats=5):
    from sklearn.base import clone
    Xv = X.values if hasattr(X, "values") else X
    aucs, accs = [], []
    for rep in range(n_repeats):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE + rep)
        for tr, va in cv.split(Xv, y, g):
            p = clone(pipe); p.fit(Xv[tr], y[tr])
            proba = p.predict_proba(Xv[va])[:, 1]
            pm = patient_aggct(g[va], y[va], proba)
            aucs.append(pm["roc_auc"]); accs.append(pm["accuracy"])
    return float(np.nanmean(aucs)), float(np.nanstd(aucs)), float(np.mean(accs)), float(np.std(accs))


## Run everything & summarise
Runs the window zoo, the patient-level zoo, and the robust CV for each pipeline, and
prints a summary table.

**Runtime note:** feature extraction over all epochs is the slow part (several minutes);
SVM and repeated CV add more. If it's too slow, comment out `SVM_RBF`, or drop
`n_repeats` in the repeated-CV calls. On a GPU-less Kaggle CPU this whole cell can take
10–20 min — that's normal.

In [8]:
# Run everything and summarise (per pipeline).
summary = []
for pname in feature_cache:
    print("\n" + "=" * 62 + f"\nPIPELINE: {pname}\n" + "=" * 62)
    wbest, wrank, wwin, wpat, wmodel = run_window_zoo(pname)
    print(f" window best = {wbest} | held-out patient AUC {wpat['roc_auc']:.3f} acc {wpat['accuracy']:.3f}")
    pbest, prank, phold, _ = run_patient_zoo(pname)
    print(f" patient best = {pbest} | held-out patient AUC {phold['roc_auc']:.3f} acc {phold['accuracy']:.3f}")
    Xw, yw, gw = get_X_y_groups(feature_cache[pname])
    mu, sd, acc, accsd = repeated_grouped_patient_cv(wmodel, Xw, yw, gw)
    print(f" ROBUST (repeated grouped CV) patient AUC {mu:.3f}+/-{sd:.3f} acc {acc:.3f}+/-{accsd:.3f}")
    summary.append({"pipeline": pname, "window_model": wbest, "patient_model": pbest,
                    "heldout_window_patient_auc": wpat["roc_auc"],
                    "heldout_patientlevel_auc": phold["roc_auc"],
                    "robust_cv_patient_auc": mu, "robust_cv_patient_auc_std": sd,
                    "robust_cv_patient_acc": acc})
summary_df = pd.DataFrame(summary).round(3)
print("\n==== SUMMARY ====")
print(summary_df.to_string(index=False))



PIPELINE: full
[full] window-level grouped-CV ROC-AUC:
   RandomForest   0.737 +/- 0.124
   LogReg         0.733 +/- 0.116
   ExtraTrees     0.730 +/- 0.129
   LightGBM       0.729 +/- 0.125
   XGBoost        0.722 +/- 0.128
   HistGB         0.721 +/- 0.123
   CatBoost       0.715 +/- 0.126
   SVM_RBF        0.711 +/- 0.112
 window best = RandomForest | held-out patient AUC 0.583 acc 0.560
[full] patient-level repeated CV (5x5): 121 patients, 796 feats
   ExtraTrees   AUC 0.758 +/- 0.106 | acc 0.671
   LightGBM     AUC 0.704 +/- 0.098 | acc 0.635
   HistGB       AUC 0.685 +/- 0.109 | acc 0.630
   LogReg_L2    AUC 0.680 +/- 0.134 | acc 0.637
   SVM_RBF      AUC 0.650 +/- 0.141 | acc 0.595
 patient best = ExtraTrees | held-out patient AUC 0.641 acc 0.520
 ROBUST (repeated grouped CV) patient AUC 0.739+/-0.079 acc 0.634+/-0.097

PIPELINE: simple
[simple] window-level grouped-CV ROC-AUC:
   RandomForest   0.750 +/- 0.080
   ExtraTrees     0.745 +/- 0.087
   CatBoost       0.735 +/- 0.080

# Patient-level stacking ensemble — the best single upgrade

Combines your strongest patient-level base learners (**LightGBM + HistGB + ExtraTrees + SVM**)
through a logistic **meta-learner**. `StackingClassifier` builds each base model's
**out-of-fold** predictions internally (`cv=5`), so the meta-learner is never trained on a
prediction a base model made about its own training data — no leak. It's evaluated with the
**same repeated 5×5 CV** as the other patient-level models and on the canonical held-out split,
then printed as a leaderboard so you can see whether stacking actually beats the best single model.

**How to use:** run this AFTER the Part 2 cells (it needs `feature_cache`, `build_patient_table`,
`make_patient_models`, `repeated_patient_cv`, `TRAIN_PIDS/TEST_PIDS`, `OPT` in memory). It runs on
whichever pipeline won the robust CV (your **simple** pipeline). If `PatientStack` tops the
leaderboard, that's your final model to report; if a single model still wins, report that instead
and note stacking didn't help — both are legitimate findings.

**Honest expectation:** stacking usually adds a *small* bump (a few points of AUC at most) or ties
the best single. A large jump would be a red flag to investigate, not a prize.

In [9]:
# ===== BEST ADD-ON: patient-level STACKING ensemble (leak-free) =====
# Combines the strongest patient-level base learners (LightGBM, HistGB, ExtraTrees,
# SVM) through a logistic meta-learner. StackingClassifier generates the base models'
# out-of-fold predictions internally (cv=5), so the meta-learner never sees a base
# prediction made on data that base model trained on. Evaluated with the SAME repeated
# stratified CV as the other patient-level models, and on the canonical held-out split.
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
import joblib

try:
    BEST_PIPE = summary_df.sort_values("robust_cv_patient_auc", ascending=False)["pipeline"].iloc[0]
except Exception:
    BEST_PIPE = "simple" if "simple" in feature_cache else list(feature_cache)[0]
print("Stacking on winning pipeline:", BEST_PIPE)

Xp, yp, pids = build_patient_table(feature_cache[BEST_PIPE])
k = min(40, Xp.shape[1])

def _base_estimators():
    est = []
    if "LightGBM" in OPT:
        est.append(("lgbm", OPT["LightGBM"](n_estimators=300, learning_rate=0.03, num_leaves=15,
                    subsample=0.8, colsample_bytree=0.6, class_weight="balanced",
                    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)))
    est.append(("hgb", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                random_state=RANDOM_STATE)))
    est.append(("et", ExtraTreesClassifier(n_estimators=600, min_samples_leaf=2,
                class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)))
    est.append(("svm", Pipeline([("sel", SelectKBest(f_classif, k=k)),
                ("clf", SVC(kernel="rbf", probability=True, class_weight="balanced",
                            random_state=RANDOM_STATE))])))
    return est

patient_stack = Pipeline([
    ("sc", StandardScaler()),
    ("clf", StackingClassifier(
        estimators=_base_estimators(),
        final_estimator=LogisticRegression(max_iter=5000, class_weight="balanced",
                                            random_state=RANDOM_STATE),
        cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
        stack_method="predict_proba", passthrough=False, n_jobs=-1)),
])

# repeated-CV estimate of the stack, next to the best singles for context
mu, sd, acc = repeated_patient_cv(patient_stack, Xp, yp)
print(f"\nPatientStack : AUC {mu:.3f} +/- {sd:.3f} | acc {acc:.3f}   (repeated 5x5 CV)")
singles = make_patient_models(k=k)
rows = [("PatientStack", mu, sd, acc)]
for nm, pipe in singles.items():
    m2, s2, a2 = repeated_patient_cv(pipe, Xp, yp)
    rows.append((nm, m2, s2, a2))
rows.sort(key=lambda r: r[1], reverse=True)
print("\nPatient-level leaderboard (repeated CV):")
for nm, m2, s2, a2 in rows:
    tag = "  <-- stack" if nm == "PatientStack" else ""
    print(f"   {nm:14s} AUC {m2:.3f} +/- {s2:.3f} | acc {a2:.3f}{tag}")

# held-out patients (canonical split)
trm = np.isin(pids, list(TRAIN_PIDS)); tem = np.isin(pids, list(TEST_PIDS))
fit_stack = clone(patient_stack).fit(Xp.values[trm], yp[trm])
pr = fit_stack.predict_proba(Xp.values[tem])[:, 1]
held = evaluate_predictions(yp[tem], (pr >= 0.5).astype(int), pr)
print("\nPatientStack held-out patients:", {k2: round(v, 3) for k2, v in held.items()})

# refit on ALL patients and save the final artifact
final_patient_stack = clone(patient_stack).fit(Xp.values, yp)
joblib.dump(final_patient_stack, "adhd_patient_stack.joblib")
pd.DataFrame({"patient_id": pids, "y_true": yp,
              "proba_oof_note": "use repeated_patient_cv for honest proba"}).to_csv(
    "adhd_patient_stack_patients.csv", index=False)
print("\nsaved adhd_patient_stack.joblib")


Stacking on winning pipeline: simple

PatientStack : AUC 0.871 +/- 0.067 | acc 0.781   (repeated 5x5 CV)

Patient-level leaderboard (repeated CV):
   LightGBM       AUC 0.874 +/- 0.068 | acc 0.799
   HistGB         AUC 0.872 +/- 0.074 | acc 0.792
   PatientStack   AUC 0.871 +/- 0.067 | acc 0.781  <-- stack
   SVM_RBF        AUC 0.823 +/- 0.067 | acc 0.729
   ExtraTrees     AUC 0.818 +/- 0.075 | acc 0.714
   LogReg_L2      AUC 0.787 +/- 0.077 | acc 0.722

PatientStack held-out patients: {'accuracy': 0.72, 'precision': 0.75, 'recall': 0.692, 'f1': 0.72, 'roc_auc': 0.788, 'mcc': 0.442}

saved adhd_patient_stack.joblib
